# EMR Assistant — ASR and diarization benchmark

Compares model combinations on **accuracy, speed and resource cost**, using the
project's own metrics so every figure is comparable with the existing report.

Combinations, in the order the supervisor asked for them:

| # | ASR | Diarization |
|---|---|---|
| 0 | Whisper `base.en` | pyannote 3.1 | *(current system — the control)* |
| 1 | Whisper `medium` | pyannote 3.1 |
| 2 | Whisper `medium` | **Sortformer** |
| 3 | **NeMo Parakeet** | pyannote 3.1 |

Plus a noise sweep on the control, since the pipeline has no preprocessing.

**Recorded for every run:** peak GPU VRAM, peak RAM, CPU seconds, model load time,
inference time, real-time factor, word accuracy, speaker accuracy.

The metric functions are imported from `scripts/evaluate_accuracy.py` in the
repository rather than reimplemented, so a number here and a number in the report
were computed by the same code.

> **From the script's own docstring:** do not tune anything against these
> recordings and re-run. That converts a measurement into a fitting exercise.
> Decide what you are reporting before running, and report every run.

**Runtime → Change runtime type → T4 GPU** before starting.

## 1. Testing hardware

This cell answers the "testing hardware" line in the report. Run it first and
keep the output — Colab does not always give you the same GPU.

In [ ]:
!pip install -q psutil

import platform, subprocess, torch, psutil, json

def hardware_report():
    info = {
        "platform": platform.platform(),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cpu_model": "unknown",
        "cpu_cores_physical": psutil.cpu_count(logical=False),
        "cpu_cores_logical": psutil.cpu_count(logical=True),
        "ram_total_gb": round(psutil.virtual_memory().total / 1e9, 1),
        "gpu": None,
        "gpu_vram_gb": None,
        "cuda": torch.version.cuda,
    }

    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                info["cpu_model"] = line.split(":", 1)[1].strip()
                break
    except Exception:
        pass

    if torch.cuda.is_available():
        info["gpu"] = torch.cuda.get_device_name(0)
        info["gpu_vram_gb"] = round(
            torch.cuda.get_device_properties(0).total_memory / 1e9, 1)

    return info

HARDWARE = hardware_report()
print(json.dumps(HARDWARE, indent=2))

{
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "python": "3.12.13",
  "torch": "2.11.0+cu128",
  "cpu_model": "Intel(R) Xeon(R) CPU @ 2.00GHz",
  "cpu_cores_physical": 1,
  "cpu_cores_logical": 2,
  "ram_total_gb": 13.6,
  "gpu": "Tesla T4",
  "gpu_vram_gb": 15.6,
  "cuda": "12.8"
}


## 2. Repository and dependencies

The evidence recordings are committed to the repo, so cloning gets the audio and
the reference scripts together. Nothing is uploaded by hand.

In [ ]:
!git clone --depth 1 https://github.com/Wajeeha-Kamran/emr-assistant-backend.git repo
%cd repo
!ls docs/evidence docs/evidence/human_distinct

Cloning into 'repo'...
remote: Enumerating objects: 184, done.
remote: Counting objects: 100% (184/184), done.
remote: Compressing objects: 100% (175/175), done.
remote: Total 184 (delta 16), reused 114 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (184/184), 23.76 MiB | 33.56 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/repo
docs/evidence:
consultation_scripts.md  human_distinct  pipeline_clip.wav  soap_heldout.md
demo_clip.wav		 load_clip.wav	 soap_expected.md   synthetic

docs/evidence/human_distinct:
consult_1.wav  consult_3.wav  diarized_output.txt
consult_2.wav  consult_4.wav  README.txt


In [ ]:
# openai-whisper is the local model, not the OpenAI API.
!pip install -q openai-whisper
!pip install -q "pyannote.audio==4.0.7"
!pip install -q soundfile
print("done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 19.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 5.4 MB/s et

### Hugging Face token

pyannote's weights are licence-gated. The token authorises the download and
nothing else. Put it in Colab's secrets (key icon, left sidebar) as `HF_TOKEN`
rather than pasting it into a cell.

Licences needed on `pyannote/segmentation-3.0`,
`pyannote/speaker-diarization-3.1` and `pyannote/speaker-diarization-community-1`.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "HF_TOKEN is empty"
print("token loaded")

token loaded


## 3. The project's own metrics

`word_error_rate` and `speaker_accuracy` come from the repository, unchanged. The
app imports inside `evaluate_accuracy.main()` are never triggered, so this needs
no database and no `.env`.

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

from scripts.evaluate_accuracy import (
    parse_scripts, normalise, strip_numerics,
    word_error_rate, speaker_accuracy, audio_duration,
    SCRIPTS_MD, TARGET,
)

scripts = parse_scripts(SCRIPTS_MD)
print(f"parsed {len(scripts)} reference scripts:", sorted(scripts))

parsed 4 reference scripts: [1, 2, 3, 4]


## 4. Measuring cost

`Measured` wraps a block and records what it consumed. Three notes on method,
because these numbers go in a report:

**CPU seconds, not CPU percent.** A percentage depends on when you sampled it.
CPU-seconds is the total work done and is reproducible.

**Peak VRAM, not final VRAM.** What matters for "will this fit on a given card"
is the high-water mark, not what was still allocated at the end.

**Load time separate from inference time.** Loading a model is a one-off cost paid
at startup; inference is paid per consultation. Averaging them together would
flatter the big models and mislead on deployment.

In [ ]:
import time, gc, threading
import torch, psutil

class Measured:
    """Context manager recording VRAM, RAM, CPU and wall time for a block."""

    def __init__(self, label):
        self.label = label
        self.stats = {}

    def __enter__(self):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        self._proc = psutil.Process()
        self._cpu0 = sum(self._proc.cpu_times()[:2])
        self._rss_peak = self._proc.memory_info().rss
        self._stop = threading.Event()

        # Sample RSS in the background: peak RAM is invisible if you only look
        # before and after.
        def sample():
            while not self._stop.wait(0.25):
                self._rss_peak = max(self._rss_peak, self._proc.memory_info().rss)

        self._sampler = threading.Thread(target=sample, daemon=True)
        self._sampler.start()
        self._t0 = time.time()
        return self

    def __exit__(self, *exc):
        self.stats["wall_s"] = round(time.time() - self._t0, 2)
        self._stop.set()
        self._sampler.join(timeout=1)

        self.stats["cpu_s"] = round(sum(self._proc.cpu_times()[:2]) - self._cpu0, 2)
        self.stats["ram_peak_gb"] = round(self._rss_peak / 1e9, 2)
        self.stats["vram_peak_gb"] = (
            round(torch.cuda.max_memory_allocated() / 1e9, 2)
            if torch.cuda.is_available() else None)
        return False

## 5. The harness

An engine is a function taking a WAV path and returning the pipeline's own shape:
`[{"text": ..., "speaker_role": "DOCTOR"|"PATIENT"}]`.

`benchmark()` loads the engine once (timed separately), then runs every script
(timed individually), and returns one row per script plus a resource summary.

In [ ]:
import pandas as pd

results = []
resource_rows = []

def benchmark(loader, engine, audio_dir, label, scripts=scripts):
    """Load once, run every script, record accuracy and cost."""
    print(f"\n=== {label} — {os.path.basename(audio_dir)} ===")

    with Measured("load") as load:
        loader()
    print(f"  model load: {load.stats['wall_s']}s, "
          f"VRAM {load.stats['vram_peak_gb']}GB")

    rows = []
    for n in sorted(scripts):
        wav = os.path.join(audio_dir, f"consult_{n}.wav")
        if not os.path.exists(wav):
            print(f"  script {n}: missing, skipped")
            continue

        ref_words, ref_spk = [], []
        for speaker, text in scripts[n]:
            w = normalise(text)
            ref_words.extend(w)
            ref_spk.extend([speaker] * len(w))

        with Measured(f"script{n}") as run:
            segments = engine(wav)

        hyp_words, hyp_spk = [], []
        for seg in segments:
            w = normalise(seg["text"])
            hyp_words.extend(w)
            hyp_spk.extend([seg["speaker_role"]] * len(w))

        wacc_nonum = max(0.0, 1 - word_error_rate(
            strip_numerics(ref_words), strip_numerics(hyp_words))) * 100
        correct, total = speaker_accuracy(ref_words, ref_spk, hyp_words, hyp_spk)
        spk = (correct / total * 100) if total else 0.0
        duration = audio_duration(wav)

        rows.append({
            "run": label,
            "audio_set": os.path.basename(audio_dir),
            "script": n,
            "word_acc": round(wacc_nonum, 1),
            "speaker_acc": round(spk, 1),
            "segments": len(segments),
            "ref_turns": len(scripts[n]),
            "audio_s": round(duration, 1),
            "infer_s": run.stats["wall_s"],
            "realtime_x": round(run.stats["wall_s"] / duration, 2) if duration else None,
            "cpu_s": run.stats["cpu_s"],
            "vram_peak_gb": run.stats["vram_peak_gb"],
            "ram_peak_gb": run.stats["ram_peak_gb"],
        })
        print(f"  script {n}: word {wacc_nonum:.1f}%  speaker {spk:.1f}%  "
              f"{run.stats['wall_s']}s  VRAM {run.stats['vram_peak_gb']}GB")

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    resource_rows.append({
        "run": label,
        "audio_set": os.path.basename(audio_dir),
        "load_s": load.stats["wall_s"],
        "load_vram_gb": load.stats["vram_peak_gb"],
        "vram_peak_gb": df.vram_peak_gb.max(),
        "ram_peak_gb": df.ram_peak_gb.max(),
        "cpu_s_mean": round(df.cpu_s.mean(), 1),
        "infer_s_mean": round(df.infer_s.mean(), 1),
        "realtime_x_mean": round(df.realtime_x.mean(), 2),
        "word_acc_mean": round(df.word_acc.mean(), 1),
        "speaker_acc_mean": round(df.speaker_acc.mean(), 1),
    })

    print(f"  MEAN  word {df.word_acc.mean():.1f}%  speaker {df.speaker_acc.mean():.1f}%"
          f"  | peak VRAM {df.vram_peak_gb.max()}GB  RT x{df.realtime_x.mean():.2f}")
    results.append(df)
    return df

## 6. Engines

### Assigning speakers

Diarization produces anonymous clusters. The project names the doctor as
**whichever speaker asks more questions** — history taking is question-driven, so
this is a majority vote across the consultation rather than a guess from who
spoke first.

Every engine below uses this same function. Change it and you stop comparing
diarization models and start comparing naming rules.

In [ ]:
import whisper
from pyannote.audio import Pipeline

_models = {}

def load_whisper(name):
    if name not in _models:
        _models[name] = whisper.load_model(name, device="cuda")
    return _models[name]

def load_pyannote():
    if "pyannote" not in _models:
        p = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1",
                                     token=os.environ["HF_TOKEN"])
        p.to(torch.device("cuda"))
        _models["pyannote"] = p
    return _models["pyannote"]


def assign_roles(segments):
    """Label the more question-asking cluster DOCTOR. Mirrors DiarizationService."""
    counts = {}
    for s in segments:
        counts[s["speaker"]] = counts.get(s["speaker"], 0) + s["text"].count("?")
    if not counts:
        return segments
    doctor = max(counts, key=counts.get)
    for s in segments:
        s["speaker_role"] = "DOCTOR" if s["speaker"] == doctor else "PATIENT"
    return segments


def words_to_turns(words, spans):
    """Give each word the speaker whose turn covers it, then merge runs."""
    def speaker_at(t):
        for start, end, spk in spans:
            if start <= t <= end:
                return spk
        # Nothing covers this moment: take the nearest turn rather than dropping
        # the word, which would silently shorten the hypothesis and flatter WER.
        return min(spans, key=lambda s: min(abs(s[0] - t), abs(s[1] - t)))[2] if spans else "A"

    merged, current = [], None
    for w in words:
        spk = speaker_at((w["start"] + w["end"]) / 2)
        if current is None or current["speaker"] != spk:
            if current:
                merged.append(current)
            current = {"speaker": spk, "text": w["word"]}
        else:
            current["text"] += w["word"]
    if current:
        merged.append(current)
    return assign_roles(merged)


def whisper_pyannote(wav, model_name):
    asr = load_whisper(model_name).transcribe(wav, word_timestamps=True)
    words = [w for seg in asr.get("segments", []) for w in seg.get("words", [])]
    turns = load_pyannote()(wav)
    spans = [(t.start, t.end, spk) for t, _, spk in turns.itertracks(yield_label=True)]
    return words_to_turns(words, spans)

ImportError: cannot import name '_center' from 'numpy._core.umath' (/usr/local/lib/python3.12/dist-packages/numpy/_core/umath.py)

## Run 0 — the control

Whisper `base.en` + pyannote: exactly what the system uses today.

**Check this before running anything else.** If word accuracy lands near
**86.4%** and speaker accuracy near **77.6%** on `human_distinct`, the notebook is
sound. If it does not, stop and find out why — every later number would inherit
the same fault.

This run also produces the GPU timing figure that Module 8.3 could never measure,
because there was no GPU.

In [ ]:
benchmark(lambda: (load_whisper("base.en"), load_pyannote()),
          lambda w: whisper_pyannote(w, "base.en"),
          "docs/evidence/human_distinct", "0. base.en + pyannote")

benchmark(lambda: None,
          lambda w: whisper_pyannote(w, "base.en"),
          "docs/evidence/synthetic", "0. base.en + pyannote (synthetic)")

## Run 1 — Whisper medium

One variable changed. ASR already meets its 85% target, so the question is what
the extra cost buys — and whether better word timings improve **speaker** accuracy
as a side effect, which is the more interesting possibility.

Watch the VRAM column. `medium` is roughly ten times the parameters of `base.en`.

In [ ]:
benchmark(lambda: load_whisper("medium"),
          lambda w: whisper_pyannote(w, "medium"),
          "docs/evidence/human_distinct", "1. medium + pyannote")

## Run 2 — Whisper medium + Sortformer

**Scaffold, not a working engine.** NeMo's install and API move between releases
and I could not verify this from outside a running Colab, so treat it as a
starting point to correct rather than code to trust.

Only the diarization call changes. Keep `words_to_turns` and `assign_roles`
exactly as they are — that is what isolates the variable.

In [ ]:
# !pip install -q "nemo_toolkit[asr]"

def load_sortformer():
    """
    Something like:

        from nemo.collections.asr.models import SortformerEncLabelModel
        _models["sortformer"] = SortformerEncLabelModel.from_pretrained(
            "nvidia/diar_sortformer_4spk-v1").to("cuda").eval()

    Check the model card for the current class name and checkpoint.
    """
    raise NotImplementedError("Verify against the NeMo docs for the installed version.")


def whisper_sortformer(wav, model_name="medium"):
    """
    Same three steps as whisper_pyannote, with step 2 swapped:

      1. Whisper transcribe, word_timestamps=True          (unchanged)
      2. Sortformer -> spans of (start, end, speaker_id)   (the only difference)
      3. words_to_turns(words, spans)                      (unchanged)

    Sortformer returns per-frame speaker activity rather than turn objects, so
    step 2 needs converting to (start, end, label) spans before step 3 will take
    it.
    """
    raise NotImplementedError

# benchmark(load_sortformer, whisper_sortformer,
#           "docs/evidence/human_distinct", "2. medium + sortformer")

## Run 3 — NeMo Parakeet

Also a scaffold. Parakeet is fast and accurate on English, but the important
detail for this project is whether it gives **word-level timestamps** — without
them, step 3 has nothing to map onto speaker turns and the diarization half of
the comparison collapses.

Check that first. If it cannot, say so in the report; that is a finding about
pipeline compatibility, not a failure to test.

In [ ]:
def load_parakeet():
    """
        from nemo.collections.asr.models import ASRModel
        _models["parakeet"] = ASRModel.from_pretrained(
            "nvidia/parakeet-tdt-0.6b-v2").to("cuda").eval()
    """
    raise NotImplementedError("Verify the current checkpoint name on the model card.")


def parakeet_pyannote(wav):
    """
    Needs word timestamps. In recent NeMo that is roughly:

        out = _models["parakeet"].transcribe([wav], timestamps=True)
        words = [{"word": w["word"], "start": w["start"], "end": w["end"]}
                 for w in out[0].timestamp["word"]]

    then the same words_to_turns(words, spans) as everything else.
    """
    raise NotImplementedError

# benchmark(load_parakeet, parakeet_pyannote,
#           "docs/evidence/human_distinct", "3. parakeet + pyannote")

## Run 4 — noise

There is no audio preprocessing in the pipeline: recordings go to Whisper exactly
as captured. This measures what that costs.

Gaussian noise at three signal-to-noise ratios — 20 dB is a quiet room, 10 dB a
busy clinic, 5 dB is bad. Fixed seed, so reruns are repeatable. Noisy copies go to
`/content`, leaving the evidence files untouched.

In [ ]:
import numpy as np, soundfile as sf

def add_noise(src_dir, dst_dir, snr_db, seed=0):
    os.makedirs(dst_dir, exist_ok=True)
    rng = np.random.default_rng(seed)
    for name in sorted(os.listdir(src_dir)):
        if not (name.startswith("consult_") and name.endswith(".wav")):
            continue
        audio, sr = sf.read(os.path.join(src_dir, name))
        noise_power = np.mean(audio ** 2) / (10 ** (snr_db / 10))
        noisy = np.clip(audio + rng.normal(0, np.sqrt(noise_power), audio.shape), -1, 1)
        sf.write(os.path.join(dst_dir, name), noisy, sr, subtype="PCM_16")
    return dst_dir

for snr in (20, 10, 5):
    add_noise("docs/evidence/human_distinct", f"/content/noise_{snr}db", snr)
print("noisy copies written")

In [ ]:
for snr in (20, 10, 5):
    benchmark(lambda: None,
              lambda w: whisper_pyannote(w, "base.en"),
              f"/content/noise_{snr}db", f"4. base.en + pyannote @ {snr}dB SNR")

## Results

Two tables. The first is the comparison the report needs; the second is per-script
detail for the appendix.

In [ ]:
import pandas as pd

detail = pd.concat([d for d in results if not d.empty], ignore_index=True)
comparison = pd.DataFrame(resource_rows)

print("HARDWARE")
for k, v in HARDWARE.items():
    print(f"  {k}: {v}")
print()

display(comparison)

detail.to_csv("/content/benchmark_detail.csv", index=False)
comparison.to_csv("/content/benchmark_comparison.csv", index=False)
print("\nwritten to /content/benchmark_detail.csv and /content/benchmark_comparison.csv")

### Recommended hardware

Derived from what was measured, not guessed. The rule below is peak VRAM plus
roughly 40% headroom, because a card sized exactly to the peak will fail the first
time a consultation runs slightly long.

In [ ]:
for row in resource_rows:
    vram = row["vram_peak_gb"] or 0
    needed = round(vram * 1.4, 1)
    rt = row["realtime_x_mean"]
    print(f"{row['run']}")
    print(f"   peak VRAM {vram} GB  ->  recommend a card with at least {needed} GB")
    print(f"   peak RAM  {row['ram_peak_gb']} GB")
    print(f"   real-time factor x{rt}  ->  a 10-minute consultation takes "
          f"about {round(rt * 600 / 60, 1)} minutes")
    print()

### Before writing any of this up

- **State the audio set with every figure.** `human_distinct` and `synthetic` are
  not comparable, and the report already says so.
- **Report every run, not the best one.**
- **Say what a win costs.** A model that gains two points of accuracy for three
  times the VRAM is a different recommendation from one that gains two points for
  free.
- **A null result is a result.** "Three alternatives were tested and the original
  held up" is a stronger finding than most people expect, and it is only
  available to you if you say so before you look.